# AI-BASED NIDS - Phase 2: Supervised Deep Learning & XAI

**Goal**: Train advanced Deep Learning models (CNN, LSTM, Transformer) to detect attacks with high precision and explain them using SHAP.

**Models**:
1. **1D-CNN**: Spatial feature extraction from flow vectors.
2. **LSTM**: Temporal sequence analysis.
3. **Transformer**: Attention-based detection.

**Explainability**: SHAP (SHapley Additive exPlanations) plots.

In [ ]:
# 1. Setup & Imports
!pip install shap

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import shap
import matplotlib.pyplot as plt

# Check GPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# 2. Load Data (Using our Loader Logic)
# Copy 'loader.py' content or upload script to Colab
# For this notebook, we simulate the loaded data structure

# Assume X_train, y_train, X_test, y_test are loaded from loader.py
# Example dimensions for 1D-CNN (samples, features, 1)

input_shape = (78, 1) # Assumed 78 features from CIC-IDS
num_classes = 1 # Binary Classification (Attack vs Benign)

print("Data Loaded (Simulated)")

In [ ]:
# 3. Model 1: 1D-CNN
def build_cnn(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn(input_shape)
cnn_model.summary()

In [ ]:
# 4. Model 2: LSTM
def build_lstm(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_lstm(input_shape)
lstm_model.summary()

In [ ]:
# 5. Model 3: Transformer (Self-Attention)
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

def build_transformer(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

transformer_model = build_transformer(input_shape, head_size=256, num_heads=4, ff_dim=4, num_transformer_blocks=4, mlp_units=[128], mlp_dropout=0.4)
transformer_model.summary()

In [ ]:
# 6. Train Models
# history_cnn = cnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
# history_lstm = lstm_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
# history_transformer = transformer_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
# cnn_model.save('cnn_model.h5')
# lstm_model.save('lstm_model.h5')
# transformer_model.save('transformer_model.h5')

In [ ]:
# 7. Explainable AI (SHAP)
# Explaining CNN predictions
# background = X_train[np.random.choice(X_train.shape[0], 100, replace=False)]
# explainer = shap.DeepExplainer(cnn_model, background)
# shap_values = explainer.shap_values(X_test[:10])

# shap.summary_plot(shap_values[0], X_test[:10], plot_type="bar")